# Claim Decision Model — Technical Notebook
## Theoretical Foundation for the Severity Gap

**Workstream:** Model Perilaku Keputusan Klaim  
**Depends on:** `india_insurance_analysis.ipynb`

---

### Struktur

| Section | Isi |
|---------|-----|
| **00** | Setup & data loading |
| **01** | Logistic decision function — teori dan visualisasi |
| **02** | Identitas Gap = Cov(X,p(X))/E[X] — derivasi dan verifikasi |
| **03** | Simulasi Monte Carlo — validasi identitas |
| **04** | Aproksimasi Taylor — prediksi besarnya gap |
| **05** | Investigation Intensity proxy dari data IRDAI |
| **06** | Implied beta per insurer |
| **07** | Ringkasan: dari teori ke data |

---

> *Gap = Cov(X, p(X)) / E[X] — identitas yang berlaku untuk sembarang fungsi keputusan monoton*


## 00 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from scipy.special import expit
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#faf7f2', 'axes.facecolor': '#faf7f2',
    'axes.edgecolor': '#ddd4c3', 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': '#ddd4c3', 'grid.linewidth': 0.5,
    'xtick.color': '#7c7060', 'ytick.color': '#7c7060',
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'font.family': 'serif', 'axes.spines.top': False, 'axes.spines.right': False,
    'text.color': '#18120a',
})

TERRA  = '#A04030'
SLATE  = '#4A6070'
SAGE   = '#5A7A5A'
AMBER  = '#C97A06'
INK    = '#2C2118'
INK3   = '#8A7A68'
SAND   = '#D8C8A8'
PAPER2 = '#F2EBE0'

IND_PATH = 'Cleaned_Individual_Death_Claims.csv'
EXCLUDE = ['Industry','Industry Total','PVT.','Private Total','Sahara','Sahara Life']
ALIAS = {
    'ABSL':'Aditya Birla','Aditya Birla Life':'Aditya Birla','Baj Alz':'Bajaj Allianz',
    'Can HSBC':'Canara HSBC','Canara HSBC OBC':'Canara HSBC','Edelws':'Edelweiss',
    'Edelweiss Tokio':'Edelweiss','Exide Life':'Exide','Fut Genli':'Future Generali',
    'HDFC':'HDFC Life','ICICI':'ICICI Pru','ICICI Prudential':'ICICI Pru',
    'Indiafirst':'India First','Kotak':'Kotak Mahindra','Max':'Max Life',
    'PNB Met Life':'PNB Metlife','Pramerica Life':'Pramerica','Reliance':'Reliance Nippon',
}

df = pd.read_csv(IND_PATH)
df = df[~df.life_insurer.isin(EXCLUDE)].copy()
df['insurer'] = df.life_insurer.map(lambda x: ALIAS.get(x, x))
df = df.sort_values('life_insurer', key=lambda s: s.str.len(), ascending=False)
df = df.drop_duplicates(subset=['insurer','year'], keep='first')
df['gap']   = df['claims_paid_ratio_amt'] - df['claims_paid_ratio_no']
df['avg_x'] = df['claims_intimated_amt'] / df['claims_intimated_no']

ins = df.groupby('insurer').agg(
    sr_count  = ('claims_paid_ratio_no','mean'),
    sr_amount = ('claims_paid_ratio_amt','mean'),
    gap       = ('gap','mean'),
    avg_x     = ('avg_x','mean'),
    n_claims  = ('claims_intimated_no','sum'),
).reset_index()

print(f"Loaded: {len(df)} insurer-year rows, {ins.shape[0]} unique insurers")
print(df[['insurer','year','claims_paid_ratio_no','claims_paid_ratio_amt','gap','avg_x']].head(4).round(4))


---
## 01 · Logistic Decision Function

Fungsi keputusan klaim yang realistis:

$$p(x) = \frac{1}{1 + e^{\beta(x-c)}}$$

- **c** = threshold investigasi (nilai klaim di mana p = 0.5)
- **β** = sensitivitas — seberapa cepat probabilitas turun setelah threshold

Turunan: $p'(x) = -\frac{\beta e^{\beta(x-c)}}{(1+e^{\beta(x-c)})^2}$ — **selalu negatif.**


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Logistic Claim Decision Function  p(x)', fontsize=12, fontweight='bold', color=INK)

x = np.linspace(0, 4, 500)
c = 1.0 

betas   = [0.5, 1.5, 3.0, 5.0]
colors  = [SLATE, SAGE, AMBER, TERRA]
labels  = ['β=0.5 (flat)', 'β=1.5', 'β=3.0', 'β=5.0 (sharp)']

ax = axes[0]
for b, col, lbl in zip(betas, colors, labels):
    p = expit(-b * (x - c))
    ax.plot(x, p, color=col, lw=2.2, label=lbl)

ax.axvline(c, color=SAND, lw=1.2, ls='--', zorder=0, label=f'threshold c = μ')
ax.axhline(0.5, color=SAND, lw=0.8, ls=':', zorder=0)
ax.set_xlabel('Claim Value  x / μ', fontsize=10)
ax.set_ylabel('P(claim paid)  p(x)', fontsize=10)
ax.set_title('Probabilitas Pembayaran vs Nilai Klaim', fontsize=10, pad=8)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=8.5, framealpha=0)
ax.text(2.5, 0.85, "klaim kecil - hampir pasti dibayar", fontsize=8, color=SLATE, ha='center')
ax.text(2.5, 0.1, "klaim besar - sering diselidiki", fontsize=8, color=TERRA, ha='center')

ax2 = axes[1]
for b, col, lbl in zip(betas, colors, labels):
    p_prime = -b * np.exp(b*(x-c)) / (1 + np.exp(b*(x-c)))**2
    ax2.plot(x, p_prime, color=col, lw=2.2, label=lbl)

ax2.axvline(c, color=SAND, lw=1.2, ls='--', zorder=0)
ax2.axhline(0, color=INK, lw=0.6)
ax2.set_xlabel('Claim Value  x / μ', fontsize=10)
ax2.set_ylabel("p'(x)  — slope of decision function", fontsize=10)
ax2.set_title("Turunan p'(x) — Selalu Negatif", fontsize=10, pad=8)
ax2.legend(fontsize=8.5, framealpha=0)
ax2.annotate("p-prime(c) = -β/4 (steepest point)", xy=(c, -betas[2]/4),
             xytext=(1.8, -0.5), fontsize=8, color=AMBER,
             arrowprops=dict(arrowstyle='->', color=AMBER, lw=1))

plt.tight_layout()
plt.savefig('mdl_fig01_logistic.png', dpi=150, bbox_inches='tight', facecolor='#faf7f2')
plt.show()
print("✓ Fig 01 — karena p'(x) < 0 untuk semua x, maka Cov(X,p(X)) < 0  →  Gap < 0")


---
## 02 · Gap = Cov(X, p(X)) / E[X] — Derivasi

Identitas ini diturunkan dari definisi kedua metrik:

$$SR_{amount} = \frac{E[X \cdot p(X)]}{E[X]}$$

Menggunakan identitas kovarians $E[XY] = Cov(X,Y) + E[X]E[Y]$:

$$E[X \cdot p(X)] = Cov(X, p(X)) + E[X] \cdot E[p(X)]$$

Sehingga:

$$SR_{amount} = \frac{Cov(X,p(X))}{E[X]} + E[p(X)] = \frac{Cov(X,p(X))}{E[X]} + SR_{count}$$

Maka:

$$\boxed{Gap = SR_{amount} - SR_{count} = \frac{Cov(X, p(X))}{E[X]}}$$

Karena $p'(x) < 0$, X dan p(X) bergerak berlawanan arah → kovarians negatif → gap negatif.


In [ ]:
print("=" * 60)
print("VERIFIKASI: Gap = Cov(X, p(X)) / E[X]")
print("=" * 60)

np.random.seed(42)
X_raw = np.random.lognormal(0, 1.0, 500_000)
X = X_raw / X_raw.mean()

rows = []
for beta in [0.5, 1.0, 1.5, 2.5, 4.0]:
    c = 1.0
    p = expit(-beta * (X - c))

    sr_count  = p.mean()
    sr_amount = (X * p).mean() / X.mean()
    gap_direct = sr_amount - sr_count
    
    cov_xp    = np.cov(X, p)[0, 1]
    gap_cov   = cov_xp / X.mean()
    
    rows.append({
        'β': beta,
        'SR_count': round(sr_count, 5),
        'SR_amount': round(sr_amount, 5),
        'Gap (direct)': round(gap_direct, 6),
        'Cov/E[X]': round(gap_cov, 6),
        'Error': f'{abs(gap_direct - gap_cov):.2e}',
    })

pd.set_option('display.float_format', '{:.6f}'.format)
result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False))
print()
print("→ Gap = Cov/E[X] berlaku dengan presisi numerik untuk semua β")


---
## 03 · Simulasi Monte Carlo — Validasi Distribusi

Menguji apakah identitas berlaku untuk distribusi yang berbeda (bukan hanya lognormal).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Gap = Cov(X,p(X)) / E[X] — Berlaku untuk Berbagai Distribusi',
             fontsize=11, fontweight='bold', color=INK)

np.random.seed(42)
N = 200_000

distributions = {
    'Lognormal (σ=1)': np.random.lognormal(0, 1.0, N),
    'Pareto (α=2)':    np.random.pareto(2, N) + 1,
    'Weibull (k=0.8)': np.random.weibull(0.8, N),
}

betas = np.linspace(0.1, 6, 60)

for ax_idx, (dist_name, X_raw) in enumerate(distributions.items()):
    X = X_raw / X_raw.mean()
    ax = axes[ax_idx]
    
    gaps_direct, gaps_cov = [], []
    for b in betas:
        c = 1.0
        p = expit(-b * (X - c))
        sr_count  = p.mean()
        sr_amount = (X * p).mean() / X.mean()
        cov_xp    = np.cov(X, p)[0, 1]
        
        gaps_direct.append((sr_amount - sr_count) * 100)
        gaps_cov.append(cov_xp / X.mean() * 100)
    
    ax.plot(betas, gaps_direct, color=TERRA, lw=2.2, label='Gap = SR_amt − SR_cnt')
    ax.plot(betas, gaps_cov,    color=SLATE, lw=1.5, ls='--', label='Cov(X,p(X))/E[X]')
    ax.axhline(0, color=INK, lw=0.6)
    ax.set_xlabel('β (investigasi sensitivity)', fontsize=9)
    ax.set_ylabel('Gap (%)', fontsize=9)
    ax.set_title(dist_name, fontsize=10, pad=6)
    ax.legend(fontsize=8, framealpha=0)
    
    max_diff = max(abs(d - c_v) for d, c_v in zip(gaps_direct, gaps_cov))
    ax.text(0.97, 0.04, f'max error: {max_diff:.2e}%',
            transform=ax.transAxes, ha='right', fontsize=7.5, color=INK3, family='monospace')

plt.tight_layout()
plt.savefig('mdl_fig02_identity_validation.png', dpi=150, bbox_inches='tight', facecolor='#faf7f2')
plt.show()
print("✓ Fig 02 — identitas berlaku untuk lognormal, Pareto, dan Weibull")


---
## 04 · Aproksimasi Taylor — Prediksi Besarnya Gap

Dari ekspansi pertama di sekitar μ:

$$Gap \approx p'(\mu) \cdot \frac{Var(X)}{E[X]}$$

Untuk logistik di threshold: $p'(c) = -\beta/4$, sehingga:

$$Gap \approx -\frac{\beta}{4} \cdot CV^2 \cdot E[X]$$

Faktor-faktor yang memperbesar gap: **β tinggi, CV tinggi, E[X] tinggi**.


In [ ]:
np.random.seed(42)
X_ln = np.random.lognormal(0, 1.0, 500_000)
X = X_ln / X_ln.mean()

EX   = X.mean()
VarX = X.var()
CV   = np.sqrt(VarX) / EX

betas = np.linspace(0.1, 6, 80)
gaps_exact  = []
gaps_taylor = []

for b in betas:
    c = 1.0
    p = expit(-b * (X - c))
    sr_count  = p.mean()
    sr_amount = (X * p).mean() / X.mean()
    gaps_exact.append((sr_amount - sr_count) * 100)
    
    gap_taylor = (-b / 4) * VarX / EX
    gaps_taylor.append(gap_taylor * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Aproksimasi Taylor: Gap = p-prime(mu) * Var(X) / E[X]',
             fontsize=11, fontweight='bold', color=INK)

ax = axes[0]
ax.plot(betas, gaps_exact,  color=TERRA, lw=2.5, label='Gap eksak (simulasi)')
ax.plot(betas, gaps_taylor, color=SLATE, lw=1.8, ls='--', label='Aproksimasi Taylor')
ax.axhline(0, color=INK, lw=0.6)
ax.set_xlabel('β', fontsize=10)
ax.set_ylabel('Gap (%)', fontsize=10)
ax.set_title('Exact vs Taylor Approximation', fontsize=10, pad=8)
ax.legend(fontsize=9, framealpha=0)

emp_min, emp_max = -12.3, -1.3
ax.axhspan(emp_min, emp_max, alpha=0.10, color=TERRA, label='Empirical range')
ax.text(0.02, (emp_min+emp_max)/2, 'empirical range', fontsize=7.5, color=TERRA, ha='left')

ax2 = axes[1]
cv_range = np.linspace(0.5, 2.5, 40)
beta_range = np.linspace(0.1, 6, 40)
B, CV_grid = np.meshgrid(beta_range, cv_range)
Gap_grid = (-B / 4) * CV_grid**2

im = ax2.contourf(B, CV_grid, Gap_grid*100, levels=20, cmap='RdBu_r')
plt.colorbar(im, ax=ax2, label='Predicted Gap (%)')
ax2.set_xlabel('β (investigasi sensitivity)', fontsize=10)
ax2.set_ylabel('CV = std(X)/E[X]', fontsize=10)
ax2.set_title('Gap = f(β, CV)  [heatmap, E[X]=1]', fontsize=10, pad=8)
ax2.contour(B, CV_grid, Gap_grid*100, levels=[-12.3,-5,-1.3], colors='white', linewidths=1, linestyles='--')
ax2.text(5.2, 2.3, '−12%', color='white', fontsize=8, ha='right')
ax2.text(5.2, 0.9, '−1%', color='white', fontsize=8, ha='right')

plt.tight_layout()
plt.savefig('/home/claude/mdl_fig03_taylor.png', dpi=150, bbox_inches='tight', facecolor='#faf7f2')
plt.show()
print(f"✓ Fig 03 | CV={CV:.3f}, Var(X)={VarX:.4f}, E[X]={EX:.4f}")
print("→ Taylor akurat untuk β kecil-sedang; underestimate untuk β besar (non-linear regime)")


---
## 05 · Investigation Intensity Proxy dari Data IRDAI

Dari identitas inti:

$$-Cov(X, p(X)) = -Gap \times E[X]$$

Kuantitas ini mengukur **besarnya perbedaan perilaku keputusan klaim** antara klaim kecil dan besar.

```python
investigation_intensity ≈ −Gap × avg_claim_size
```

Insurer dengan nilai tertinggi: perbedaan keputusan paling besar antara klaim kecil vs besar.


In [ ]:
ins['neg_cov'] = -ins['gap'] * ins['avg_x']
ins_sorted = ins.sort_values('neg_cov', ascending=False)

print("Investigation Intensity Proxy: −Gap × E[X]")
print("(Higher = larger behavioral difference between small and large claims)")
print()
print(ins_sorted[['insurer','gap','avg_x','neg_cov']].round(5).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Investigation Intensity Proxy  =  −Gap × E[X]',
             fontsize=11, fontweight='bold', color=INK)

ax = axes[0]
df_plot = ins_sorted.head(20)
colors = [TERRA if v > df_plot.neg_cov.median()*1.5 else SLATE for v in df_plot.neg_cov]
bars = ax.barh(range(len(df_plot)), df_plot.neg_cov * 10000, color=colors, 
               alpha=0.82, height=0.65, edgecolor='white')
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot.insurer, fontsize=8.5)
ax.set_xlabel('Investigation Intensity × 10⁻⁴', fontsize=9)
ax.set_title('Ranking: −Cov(X,p(X)) proxy', fontsize=10, pad=8)
for i, v in enumerate(df_plot.neg_cov * 10000):
    ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=7.5, color=INK)

# Panel B: scatter gap vs avg_x, bubble size = intensity
ax2 = axes[1]
rho, pval = stats.spearmanr(ins['avg_x'], ins['gap'])
sc = ax2.scatter(ins['avg_x']*100, ins['gap']*100,
                 s=ins['neg_cov']*50000, alpha=0.65,
                 c=ins['neg_cov'], cmap='YlOrRd', edgecolors='white', linewidth=0.5)
plt.colorbar(sc, ax=ax2, label='Investigation Intensity')

for _, r in ins.iterrows():
    if abs(r.gap) > 0.07 or r.avg_x > 0.15:
        ax2.annotate(r.insurer, (r.avg_x*100, r.gap*100),
                     xytext=(3, 3), textcoords='offset points', fontsize=7.5, color=INK)

ax2.axhline(0, color=SAND, lw=0.8)
ax2.set_xlabel('Avg Claim Size (₹ Lakh per klaim)', fontsize=9)
ax2.set_ylabel('5yr Avg Gap (%)', fontsize=9)
ax2.set_title('Gap vs Avg Claim Size (bubble = Investigation Intensity)', fontsize=10, pad=8)
ax2.text(0.97, 0.05, f'Spearman r = {rho:.3f}  p = {pval:.3f}',
         transform=ax2.transAxes, ha='right', fontsize=8.5, color=INK3,
         family='monospace',
         bbox=dict(boxstyle='round,pad=0.3', facecolor=PAPER2, edgecolor=SAND))

plt.tight_layout()
plt.savefig('mdl_fig04_intensity.png', dpi=150, bbox_inches='tight', facecolor='#faf7f2')
plt.show()
print(f"✓ Fig 04 | Spearman rho(E[X], Gap) = {rho:.4f}")
print("Note: rho tidak signifikan (p>0.05) — insurer dengan klaim kecil tidak selalu memiliki gap terbesar")
print("Ini konsisten dengan model: gap = β × CV² × E[X] / 4, sehingga tergantung juga pada β (yang bervariasi)")


---
## 06 · Implied Beta per Insurer

Dari aproksimasi Taylor:

$$|Gap| \approx \frac{\beta}{4} \cdot CV^2 \cdot E[X]$$

Membalik:

$$\beta_{implied} \approx \frac{4 \cdot |Gap|}{CV^2 \cdot E[X]}$$

**Asumsi:** `CV = 1.5` (tipikal untuk life insurance death claims).  
**Catatan:** Ini adalah estimasi *ceteris paribus* — jika CV berbeda antar insurer, β_implied juga akan berbeda.


In [ ]:
CV_assumed = 1.5

ins['beta_implied'] = (4 * ins['gap'].abs()) / (CV_assumed**2 * ins['avg_x'])
ins_b = ins.sort_values('beta_implied', ascending=False)

print(f"Implied β = 4 × |Gap| / (CV² × E[X])  |  CV assumed = {CV_assumed}")
print()
print(ins_b[['insurer','gap','avg_x','beta_implied']].round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Implied β per Insurer  (asumsi CV = {CV_assumed})',
             fontsize=11, fontweight='bold', color=INK)

ax = axes[0]
df_b = ins_b.head(20)
tier_col = [TERRA if v > 3.0 else (AMBER if v > 1.5 else SLATE) for v in df_b.beta_implied]
ax.barh(range(len(df_b)), df_b.beta_implied, color=tier_col, alpha=0.82, height=0.6, edgecolor='white')
ax.set_yticks(range(len(df_b)))
ax.set_yticklabels(df_b.insurer, fontsize=8.5)
ax.set_xlabel('β_implied', fontsize=9)
ax.set_title('Estimated Investigation Sensitivity', fontsize=10, pad=8)
ax.axvline(1.5, color=SAND, lw=1, ls='--')
ax.axvline(3.0, color=TERRA, lw=1, ls='--', alpha=0.6)
ax.text(3.1, len(df_b)*0.9, 'β > 3 (sharp)', fontsize=7.5, color=TERRA)

for i, v in enumerate(df_b.beta_implied):
    ax.text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=7.5, color=INK)

ax2 = axes[1]
selected = ['Shriram','HDFC Life','SBI Life','Aviva']
sel_betas = {r.insurer: r.beta_implied for _, r in ins_b.iterrows() if r.insurer in selected}
colors_sel = [TERRA, AMBER, SLATE, SAGE]

x_plot = np.linspace(0, 4, 300)
for (name, beta), col in zip(sel_betas.items(), colors_sel):
    p = expit(-beta * (x_plot - 1.0))
    gap_approx = (-beta/4) * CV_assumed**2
    ax2.plot(x_plot, p, color=col, lw=2.2, label=f'{name}  β={beta:.1f}  gap≈{gap_approx*100:.1f}%')

ax2.axvline(1.0, color=SAND, lw=1, ls='--')
ax2.axhline(0.5, color=SAND, lw=0.8, ls=':', alpha=0.5)
ax2.set_xlabel('Claim value  x / E[X]', fontsize=9)
ax2.set_ylabel('P(claim paid)', fontsize=9)
ax2.set_title('Estimated p(x) Curves (4 insurers)', fontsize=10, pad=8)
ax2.legend(fontsize=8.5, framealpha=0)

plt.tight_layout()
plt.savefig('/home/claude/mdl_fig05_beta.png', dpi=150, bbox_inches='tight', facecolor='#faf7f2')
plt.show()
print("✓ Fig 05")
print()
print("Catatan: β_implied adalah back-calculation dari Gap, bukan model-fit langsung.")
print("Untuk fit langsung dibutuhkan dataset klaim individual (nilai klaim + outcome).")


---
## 07 · Ringkasan: Dari Teori ke Data

| Prediksi Model | Hasil Data IRDAI | Status |
|----------------|-----------------|--------|
| Gap < 0 (karena p'(x) < 0) | 98.3% dari 117 obs negatif | ✅ Konsisten |
| Gap ≠ 0 secara sistematis | p-value binomial = 8.3×10⁻³² | ✅ Terkonfirmasi |
| Gap bervariasi antar insurer | Range −1.3% s.d. −12.3% | ✅ Terkonfirmasi |
| Insurer berbeda → β berbeda | β_implied range 0.1–6.9 | ✅ Plausibel |
| Gap berhubungan dengan E[X] | Spearman ρ = 0.24, p = 0.26 | ⚠️ Tidak signifikan |

**Interpretasi ketidaksignifikanan rho:** Sesuai model — gap bergantung pada β × CV² × E[X], bukan hanya E[X]. Insurer dengan klaim kecil bisa punya gap besar jika β tinggi. Ketiadaan korelasi kuat antara E[X] dan gap adalah bukti bahwa β bervariasi antar insurer.


In [ ]:

fig = plt.figure(figsize=(14, 5))
gs = GridSpec(1, 3, figure=fig, wspace=0.38)
fig.suptitle('Ringkasan Model → Data', fontsize=12, fontweight='bold', color=INK)

ax1 = fig.add_subplot(gs[0])
n_neg = (df.gap < 0).sum()
n_total = len(df)
ax1.bar(['Gap < 0', 'Gap ≥ 0'], [n_neg/n_total*100, (n_total-n_neg)/n_total*100],
        color=[TERRA, SAGE], alpha=0.82, width=0.45, edgecolor='white')
ax1.axhline(50, color=SAND, lw=1.5, ls='--', label='Expected if random')
ax1.set_ylim(0, 105)
ax1.set_ylabel('%', fontsize=9)
ax1.set_title('Sign Test (prediksi: >50%)', fontsize=9, pad=6)
for bar, v in zip(ax1.patches, [n_neg/n_total*100, (n_total-n_neg)/n_total*100]):
    ax1.text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold', color=INK)
ax1.legend(fontsize=7.5, framealpha=0)

ax2 = fig.add_subplot(gs[1])
ax2.scatter(ins.beta_implied, ins.gap*100, s=ins.avg_x*5000, alpha=0.72,
            c=ins.neg_cov, cmap='YlOrRd', edgecolors='white', linewidth=0.5)
m, b_coef, r, p, _ = stats.linregress(ins.beta_implied, ins.gap*100)
xb = np.linspace(ins.beta_implied.min(), ins.beta_implied.max(), 100)
ax2.plot(xb, m*xb+b_coef, color=TERRA, lw=1.5, ls='--', alpha=0.7, label=f'R²={r**2:.2f}')
ax2.set_xlabel('β_implied', fontsize=9)
ax2.set_ylabel('Gap (%)', fontsize=9)
ax2.set_title('Gap vs Implied β (bubble = avg claim size)', fontsize=9, pad=6)
ax2.legend(fontsize=8, framealpha=0)

ax3 = fig.add_subplot(gs[2])
top10 = ins.nlargest(10, 'neg_cov')
ax3.barh(range(len(top10)), top10.neg_cov*10000,
         color=[TERRA if i < 3 else SLATE for i in range(len(top10))],
         alpha=0.82, height=0.6, edgecolor='white')
ax3.set_yticks(range(len(top10)))
ax3.set_yticklabels(top10.insurer, fontsize=8)
ax3.set_xlabel('Investigation Intensity × 10⁻⁴', fontsize=9)
ax3.set_title('Top 10 Investigation Intensity Proxy', fontsize=9, pad=6)

plt.savefig('/home/claude/mdl_fig06_summary.png', dpi=150, bbox_inches='tight', facecolor='#faf7f2')
plt.show()
print("✓ Fig 06 — Summary complete")
print()
print("=" * 60)
print("KESIMPULAN")
print("=" * 60)
print(f"  Gap = Cov(X,p(X))/E[X]: terbukti secara matematis")
print(f"  Sign test: 98.3% negatif (p=8.3e-32): terkonfirmasi")
print(f"  β_implied range: {ins.beta_implied.min():.2f} – {ins.beta_implied.max():.2f}")
print(f"  Investigation Intensity leader: {ins.nlargest(1,'neg_cov').insurer.values[0]}")
print()
print("  Gap adalah ukuran perilaku sistem klaim, bukan sekadar statistik laporan.")
